In [2]:
import os
import pandas as pd
import time
import json
import pytesseract
import re
import matplotlib.pyplot as plt
import numpy as np
import cv2
import shutil
from PIL import Image, ImageEnhance, ImageFilter
import pytesseract  


<h1>Créé un dossier contenant les informations texte sur les images</h1>

In [4]:
def traiter_image(image_path, zone_texte, contraste=4):
    img = Image.open(image_path)
    
    img_crop = img.crop(zone_texte)

    img_crop_gray = img_crop.convert('L')

    enhancer = ImageEnhance.Contrast(img_crop_gray)
    img_crop_contraste = enhancer.enhance(contraste)

    texte = pytesseract.image_to_string(img_crop_contraste, lang='fra+eng')

    # # Afficher les images à chaque étape
    # plt.figure(figsize=(10, 6))

    # plt.subplot(1, 3, 1)
    # plt.title('Image Originale Recadrée')
    # plt.imshow(img_crop)
    # plt.axis('off')

    # plt.subplot(1, 3, 2)
    # plt.title('Niveaux de Gris')
    # plt.imshow(img_crop_gray, cmap='gray')
    # plt.axis('off')

    # plt.subplot(1, 3, 3)
    # plt.title('Contraste Amélioré')
    # plt.imshow(img_crop_contraste, cmap='gray')
    # plt.axis('off')

    # plt.tight_layout()
    # plt.show()

    return texte

def extraire_informations(texte):
    informations = {
        "drive_number": None,
        "gps": {
            "latitude": None,
            "longitude": None,
            "altitude": None
        },
        "position_km": None,
        "date": None,
        "time": None,
        "gps_quality": None,
        "width": None,
        "height": None,
        "speed_kmh": None,
        "additional_info": {
            "gebied": None,
            "geo_code": None,
            "turnout_number": None,
            "direction": None,
            "emplacement": None,
            "route": None
        }
    }
    
    try:
        # Drive No.
        drive_match = re.search(r'Drive No\.\s*(\d+)', texte)
        if drive_match:
            informations["drive_number"] = drive_match.group(1)

        # GPS Latitude
        latitude_match = re.search(r'GPS Latitude\s*([-\d.,]+)', texte)
        if latitude_match:
            informations["gps"]["latitude"] = float(latitude_match.group(1).replace(',', '.'))

        # Position
        position_match = re.search(r'Position\s*(.*?)\s*GPS', texte, re.DOTALL)
        if position_match:
            position_value = position_match.group(1)  # Récupère tout entre "Position" et "GPS"
            
            # Remplace les tirets longs par un simple tiret et supprime les espaces
            position_value = position_value.replace('—-', '-').replace('-', '-').replace('—','-').replace('kn','km').strip()

            print(position_value)  # Pour vérifier la valeur extraite
                # Vérifie si la valeur restante peut être convertie en float
            if position_value:
                try:
                    informations["position_km"] = float(position_value.replace(',', '.'))  
                except ValueError:
                    informations["position_km"] = None  
            else:
                informations["position_km"] = None  
        else:
            informations["position_km"] = None  



        # GPS Longitude
        longitude_match = re.search(r'GPS Longitude\s*([-\d.,\s]+)', texte)
        if longitude_match:
            longitude_value = longitude_match.group(1).replace(',', '.').strip().replace(' ', '')
            informations["gps"]["longitude"] = float(longitude_value)

        # Altitude
        altitude_match = re.search(r'GPS Altitude\s*([-\d.,]+)', texte)
        if altitude_match:
            informations["gps"]["altitude"] = float(altitude_match.group(1).replace(',', '.'))

        # Date
        date_match = re.search(r'Date\s*([\d/]+)', texte)
        if date_match:
            informations["date"] = date_match.group(1)

        # Time
        time_match = re.search(r'Tine\s*([\d:.,]+)', texte)
        if time_match:
            informations["time"] = time_match.group(1)

        # GPS Quality
        quality_match = re.search(r'GPS Quality\s*([\d.]+)', texte)
        if quality_match:
            informations["gps_quality"] = float(quality_match.group(1))

        # Width
        width_match = re.search(r'Width\s*([^\n]*)', texte)
        if width_match:
            width_value = width_match.group(1).strip()
            informations["width"] = "0mm" if "Onn" or "O nn" or "O mm" in width_value else width_value

        # Height
        height_match = re.search(r'Heigth\s*([^\n]*)', texte)
        if height_match:
            height_value = height_match.group(1).strip()
            informations["height"] = "0mm" if "Onn" in height_value or "O nn" or "O mm" in height_value else height_value

        # Speed
        speed_match = re.search(r'Speed\s*([-\d.,]+)\s*k(?:mh|wh)', texte)
        if speed_match:
            informations["speed_kmh"] = float(speed_match.group(1).replace(',', '.'))

        # Additional Info
        additional_info_matches = {
            "gebied": re.search(r'Gebied\s*([^\n]*)', texte),
            "geo_code": re.search(r'Geo Code[:\s]*([^\n]*)', texte),
            "turnout_number": re.search(r'Turnout #:\s*([^\n]*)', texte),
            "direction": re.search(r'Direction:\s*([^\n]*)', texte),
            "emplacement": re.search(r'Enplacenent\s*([^\n]*)', texte),
            "route": re.search(r'Route:\s*([^\n]*)', texte),
        }

        for key, match in additional_info_matches.items():
            if match:
                informations["additional_info"][key] = match.group(1).strip()
            else:
                informations["additional_info"][key] = "Unknown"  
        
    except ValueError as e:
        print(f"Erreur lors de l'extraction des informations : {e}")
    except Exception as e:
        print(f"Erreur générale : {e}")
    
    return informations

def parcourir_repertoire(repertoire, zone_texte, repertoire_sortie):
    if not os.path.exists(repertoire_sortie):
        os.makedirs(repertoire_sortie)
    
    for fichier in os.listdir(repertoire):
        if fichier.lower().endswith(('.jpg', '.jpeg', '.png')):
            chemin_image = os.path.join(repertoire, fichier)
            

            texte = traiter_image(chemin_image, zone_texte)
            

            resultat = {
                'nom_image': fichier,
                'texte_reconnu': texte
            }


            informations = extraire_informations(texte)
            resultat['informations'] = informations
            

            nom_fichier_json = os.path.splitext(fichier)[0] + '.json'
            chemin_json = os.path.join(repertoire_sortie, nom_fichier_json)
            with open(chemin_json, 'w', encoding='utf-8') as json_file:
                json.dump(resultat, json_file, ensure_ascii=False, indent=4)

In [6]:
repertoire_images = "/Volumes/Seagate Drive/pdi/nouvelle_tournee/9820_up"
repertoire_sortie = "/Volumes/Seagate Drive/pdi/nouvelle_tournee/sortie_rc"
zone_texte = (0, 0, 420, 180)

parcourir_repertoire(repertoire_images, zone_texte, repertoire_sortie)

print("Traitement terminé.")

-0,396km
-0,393km
-0,390km
-0.387km
-0.384km
-0.381km
1,002km
Erreur lors de l'extraction des informations : could not convert string to float: '7..7317600'
-0.378km
-0.375km
-0.375km
-0,369km
-0.366km
-0.366km
-0.360km
-0.357km
-0.357km
-0.351km
1,005km
-0,348km
-0.345km
-0.342km
-0,339km


KeyboardInterrupt: 

<h1>Créé un json unique regroupant les informations odométriques</h1>

In [45]:
import os
import json
import re

# Chemins des fichiers
dossier_json = "/Volumes/Seagate Drive/pdi/nouvelle_tournee/sortie_rc"
fichier_sortie = "/Volumes/Seagate Drive/pdi/nouvelle_tournee/sortie_rc_json_odo/odo_extraits.json"

# Regex pour la position (tolère espace avant "km", accepte "," ou ".", et gère les nombres négatifs)
position_pattern = re.compile(r"Position\s+(-?[\d]+[.,]\d+)\s*k", re.IGNORECASE)

# Liste des résultats
positions = []

# Parcours des fichiers JSON dans le dossier
for fichier in os.listdir(dossier_json):
    if fichier.endswith(".json"):  # Vérifie que c'est un fichier JSON
        chemin_fichier = os.path.join(dossier_json, fichier)

        if os.path.getsize(chemin_fichier) == 0:  # Ignore les fichiers vides
            print(f"⚠️  Fichier vide ignoré : {fichier}")
            continue  

        with open(chemin_fichier, "r", encoding="utf-8") as f:
            try:
                data = json.load(f)  # Charge le fichier JSON
                texte = data.get("texte_reconnu", "")

                if "texte_reconnu" not in data:
                    print(f"⚠️  Clé 'texte_reconnu' absente dans {fichier}")

                # Recherche de la position
                match_position = position_pattern.search(texte)

                # Extraction et conversion de la valeur
                if match_position:
                    position_km = match_position.group(1).replace(",", ".")
                    # Supprimer le signe '-' si présent
                    position_km = position_km.replace("-", "")
                    positions.append({"fichier": fichier, "position_km": float(position_km)})
                else:
                    print(f"📏 Aucune position trouvée dans {fichier}")

            except json.JSONDecodeError:
                print(f"❌ Erreur JSON dans {fichier}")

# Sauvegarde en JSON
with open(fichier_sortie, "w", encoding="utf-8") as f_out:
    json.dump(positions, f_out, indent=4, ensure_ascii=False)

print(f"✅ {len(positions)} fichiers traités et sauvegardés dans {fichier_sortie}")

📏 Aucune position trouvée dans 240902_9820_RC_10299000.json
📏 Aucune position trouvée dans 240902_9820_RC_10347000.json
📏 Aucune position trouvée dans 240902_9820_RC_10350000.json
📏 Aucune position trouvée dans 240902_9820_RC_10353000.json
📏 Aucune position trouvée dans 240902_9820_RC_10356000.json
📏 Aucune position trouvée dans 240902_9820_RC_10359000.json
📏 Aucune position trouvée dans 240902_9820_RC_10362000.json
📏 Aucune position trouvée dans 240902_9820_RC_10365000.json
📏 Aucune position trouvée dans 240902_9820_RC_10368000.json
📏 Aucune position trouvée dans 240902_9820_RC_10371000.json
📏 Aucune position trouvée dans 240902_9820_RC_10374000.json
📏 Aucune position trouvée dans 240902_9820_RC_10377000.json
📏 Aucune position trouvée dans 240902_9820_RC_10380000.json
📏 Aucune position trouvée dans 240902_9820_RC_10383000.json
📏 Aucune position trouvée dans 240902_9820_RC_10386000.json
📏 Aucune position trouvée dans 240902_9820_RC_10389000.json
📏 Aucune position trouvée dans 240902_98

<h1>Donne la liste des fichiers correspondants à l'odométrie recherchée</h1>

In [8]:
import json

# Chemin du fichier JSON contenant les positions en kilomètres
fichier_odo = "/Volumes/Seagate Drive/pdi/nouvelle_tournee/sortie_rc_json_odo/odo_extraits.json"
#10000000
def trouver_fichiers_par_odo(odo_recherche):
    """
    Recherche les fichiers contenant une position en kilomètres proche de la valeur odo_recherche (en centièmes de millimètres).
    La recherche se fait à ±1 km autour de la valeur convertie.
    """
    try:
        # Convertir odo_recherche (en centièmes de millimètres) en kilomètres
        odo_km = odo_recherche / 10000000  # Conversion

        # Définir l'intervalle de recherche (±1 km)
        intervalle_min = odo_km - 0.001
        intervalle_max = odo_km + 0.001

        # Charger le fichier JSON contenant les positions
        with open(fichier_odo, "r", encoding="utf-8") as f:
            odo_data = json.load(f)

        # Liste pour stocker les fichiers correspondants
        fichiers_correspondants = []

        # Parcourir les données du JSON
        for item in odo_data:
            # Extraire la position en kilomètres
            position_km = item.get("position_km")

            # Vérifier si la position est dans l'intervalle
            if position_km is not None and intervalle_min <= position_km <= intervalle_max:
                # Remplacer .json par .jpg dans le nom du fichier
                fichier_jpg = item["fichier"].replace(".json", ".jpg")
                fichiers_correspondants.append(fichier_jpg)

        return fichiers_correspondants

    except FileNotFoundError:
        print("Le fichier des positions n'existe pas.")
        return []
    except json.JSONDecodeError:
        print("Erreur de lecture du fichier JSON.")
        return []

In [10]:
# Exemple d'utilisation
odo_recherche = 34489480.0  # Exemple de valeur en centièmes de millimètres
fichiers_trouves = trouver_fichiers_par_odo(odo_recherche)

# Créer une liste pour stocker les fichiers correspondants
liste_fichiers_correspondants = []

if fichiers_trouves:
    print("Fichiers correspondants trouvés :")
    for fichier in fichiers_trouves:
        print(fichier)
        liste_fichiers_correspondants.append(fichier)  # Ajouter le fichier à la liste
else:
    print("Aucun fichier correspondant trouvé.")

# Afficher la liste des fichiers correspondants
print("Liste des fichiers correspondants :", liste_fichiers_correspondants)

Fichiers correspondants trouvés :
240902_9820_RC_17262000.jpg
240902_9820_RC_18264000.jpg
240902_9820_RC_18672000.jpg
Liste des fichiers correspondants : ['240902_9820_RC_17262000.jpg', '240902_9820_RC_18264000.jpg', '240902_9820_RC_18672000.jpg']


In [77]:
from PIL import Image
import os

# Chemin du dossier contenant les images
dossier_images = "/Volumes/Seagate Drive/pdi/nouvelle_tournee/9820_up"

# Liste des noms d'images à afficher
liste_images = liste_fichiers_correspondants

# Parcourir la liste des images et les afficher
for nom_image in liste_images:
    # Construire le chemin complet de l'image
    chemin_image = os.path.join(dossier_images, nom_image)
    
    # Vérifier si le fichier existe
    if os.path.exists(chemin_image):
        # Charger et afficher l'image
        image = Image.open(chemin_image)
        image.show()
    else:
        print(f"⚠️ Le fichier {nom_image} n'existe pas dans le dossier {dossier_images}.")

In [14]:
odo_recherche = 37391910.0 # Exemple de valeur en centièmes de millimètres
fichiers_trouves = trouver_fichiers_par_odo(odo_recherche)

# Créer une liste pour stocker les fichiers correspondants
liste_fichiers_correspondants_2 = []

if fichiers_trouves:
    print("Fichiers correspondants trouvés :")
    for fichier in fichiers_trouves:
        print(fichier)
        liste_fichiers_correspondants_2.append(fichier)  # Ajouter le fichier à la liste
else:
    print("Aucun fichier correspondant trouvé.")

# Afficher la liste des fichiers correspondants
print("Liste des fichiers correspondants :", liste_fichiers_correspondants_2)

Fichiers correspondants trouvés :
240902_9820_RC_17553000.jpg
240902_9820_RC_17556000.jpg
240902_9820_RC_46239000.jpg
240902_9820_RC_46242000.jpg
240902_9820_RC_46245000.jpg
240902_9820_RC_78213000.jpg
Liste des fichiers correspondants : ['240902_9820_RC_17553000.jpg', '240902_9820_RC_17556000.jpg', '240902_9820_RC_46239000.jpg', '240902_9820_RC_46242000.jpg', '240902_9820_RC_46245000.jpg', '240902_9820_RC_78213000.jpg']


In [71]:
liste_images_2 = liste_fichiers_correspondants_2

for nom_image in liste_images_2:
    # Construire le chemin complet de l'image
    chemin_image = os.path.join(dossier_images, nom_image)
    
    # Vérifier si le fichier existe
    if os.path.exists(chemin_image):
        # Charger et afficher l'image
        image = Image.open(chemin_image)
        image.show()
    else:
        print(f"⚠️ Le fichier {nom_image} n'existe pas dans le dossier {dossier_images}.")

In [65]:
odo_recherche = 36958880.0 # Exemple de valeur en centièmes de millimètres
fichiers_trouves = trouver_fichiers_par_odo(odo_recherche)

# Créer une liste pour stocker les fichiers correspondants
liste_fichiers_correspondants_3 = []

if fichiers_trouves:
    print("Fichiers correspondants trouvés :")
    for fichier in fichiers_trouves:
        print(fichier)
        liste_fichiers_correspondants_3.append(fichier)  # Ajouter le fichier à la liste
else:
    print("Aucun fichier correspondant trouvé.")

# Afficher la liste des fichiers correspondants
print("Liste des fichiers correspondants :", liste_fichiers_correspondants_3)

Fichiers correspondants trouvés :
240902_9820_RC_18018000.jpg
240902_9820_RC_18969000.jpg
240902_9820_RC_78258000.jpg
Liste des fichiers correspondants : ['240902_9820_RC_18018000.jpg', '240902_9820_RC_18969000.jpg', '240902_9820_RC_78258000.jpg']


In [73]:
liste_images_3 = liste_fichiers_correspondants_3

for nom_image in liste_images_3:
    # Construire le chemin complet de l'image
    chemin_image = os.path.join(dossier_images, nom_image)
    
    # Vérifier si le fichier existe
    if os.path.exists(chemin_image):
        # Charger et afficher l'image
        image = Image.open(chemin_image)
        image.show()
    else:
        print(f"⚠️ Le fichier {nom_image} n'existe pas dans le dossier {dossier_images}.")